# 03_01 Simple attention

*📖 Book: section 3.3*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** How can a model weigh the earlier letters and mix them into one summary? First with nothing to learn: just scores, softmax and a weighted average.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to score how well two letters match, using the dot product
- How softmax turns scores into attention weights that add up to 1
- How the weights mix the letters into one context vector
- Why attention with nothing to learn always prefers each letter itself

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **attention score** | a number for how well two letters match: the dot product of their rows | `δ` against `δ` scores 2 |
| **attention weights** | the scores after softmax: one share per earlier letter, and the shares add up to 1 | `[0.21, 0.21, 0.58]` |
| **context vector** | the mix of all the letters' rows, each counted by its attention weight | `[0.79, 0.79]` in our tiny example |
| **softmax** | turns any list of scores into shares between 0 and 1 that add up to 1 (see [A_04](../A_toolbox/A_04_pytorch_in_this_repo.ipynb)) | `[1, 1, 2]` → `[0.21, 0.21, 0.58]` |

## 3. The idea

### ❓ What does attention do, in one picture?

Take the start of `ἄνδρα`, the three letters `ἄ ν δ`. The model has read them and is about to guess the next one. It looks back at each letter and decides how much to listen to it. Each letter gets a **share**, and the shares add up to 1 (all of its focus). Then it blends the three letters, each counted by its share.

```
look at:     ἄ        ν        δ
share:       ?        ?        ?         (the three add up to 1)
blend:   share×ἄ  +  share×ν  +  share×δ   =  the context vector
```

So three steps: score each letter, turn the scores into shares, blend. Here the letters have nothing to learn. We use their embedding rows from [02_05](../02_text_to_numbers/02_05_token_embeddings.ipynb) just as they are.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how do we score the letters?

The score is a **dot product**: multiply two rows number by number and add up the results ([A_03](../A_toolbox/A_03_dot_product.ipynb) explains it with films; if you skipped it, 10 minutes there will make this whole chapter easier). A high score means the two rows point the same way.

To keep the sums small, we give each letter an invented row of just 2 numbers. The letter doing the looking is `δ`:

| Letter | Row | The maths (`δ` = `[1, 1]`) | Score |
|---|---|---|---|
| `ἄ` | `[1, 0]` | (1 × 1) + (1 × 0) | **1** |
| `ν` | `[0, 1]` | (1 × 0) + (1 × 1) | **1** |
| `δ` | `[1, 1]` | (1 × 1) + (1 × 1) | **2** |

The code scores all three at once with `X @ X[2]`. The `@` is **matrix multiplication**, and for a table and one row it means: take the dot product of every row of the table with that row. Here `X` is the table of three rows, `X[2]` is the row of `δ`, and the result is three scores, one per row of `X`.

👉 **Predict first:** will it print `[1, 1, 2]`?

In [ ]:
import torch

letters = ["ἄ", "ν", "δ"]
X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # one invented row per letter

scores = X @ X[2]  # every row dotted with the row of δ (the 3rd letter)
print(scores)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`tensor([1., 1., 2.])`, the same as the table. `X[2]` is the 3rd row (counting starts at 0), and `X @ X[2]` dots every row of `X` with it.

</details>

### ❓ Step 2: how do the scores become shares?

Scores like `[1, 1, 2]` aren't shares yet, because they don't add up to 1. **Softmax** fixes that ([A_04](../A_toolbox/A_04_pytorch_in_this_repo.ipynb)). It makes every score positive, keeps the order (bigger score, bigger share), and divides so the total is 1. What comes out are the attention weights.

👉 **Predict first:** `δ` has the biggest score, 2. Will its share be about half, or more? And what will the three shares add up to?

In [ ]:
weights = torch.softmax(scores, dim=0)
for letter, w in zip(letters, weights, strict=True):
    print(f"{letter}  share {w:.3f}")
print("they add up to", weights.sum().item())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`0.212`, `0.212` and `0.576`. So `δ` gets a bit more than half, and the three add up to 1 (`1.0`, apart from tiny rounding in the last digit). Notice that a score of 2 against 1 gave a share 2.7 times bigger, not 2 times. Softmax boosts the biggest.

</details>

### ❓ Step 3: how do we blend the letters?

Multiply each letter's row by its share, then add them up. The blended row is the **context vector**:

```
0.212 × [1, 0]  +  0.212 × [0, 1]  +  0.576 × [1, 1]
=  [0.212, 0]  +  [0, 0.212]  +  [0.576, 0.576]
=  [0.788, 0.788]
```

In PyTorch this whole blend is one `@`: the row of 3 shares dotted with each of the 2 columns of `X`.

👉 **Predict first:** will `weights @ X` print `[0.788, 0.788]`?

In [ ]:
context = weights @ X
print(context)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`tensor([0.7881, 0.7881])`. It leans towards `δ`'s row `[1, 1]`, because `δ` got the biggest share.

</details>

**Look at:** the width of each arrow. A wider arrow is a bigger share, and the three arrows meet in the context vector.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

share = weights.tolist()
fig = studio_figure("A blend with shares", "δ gets the widest arrow")
ax = diagram_axes(fig)
tag(ax, 5, 5.55, "the letters it looks at", MUTED, 9)
for i, letter in enumerate("ἄνδ"):
    cx = 2.2 + 2.8 * i
    color = ACCENT if i == 2 else MUTED
    box(ax, cx, 4.7, 1.7, 0.9, f"{letter}\n{X[i].int().tolist()}", color, mono=True)
    arrow(
        ax,
        (cx, 3.65),
        (3.9 + 1.1 * i, 1.85),
        color=color,
        width=1 + 10 * share[i],
        head=12,
    )
    tag(ax, cx, 4.0, f"share {share[i]:.2f}", color, 10, mono=True)
shown = ", ".join(f"{v:.3f}" for v in context.tolist())
box(ax, 5, 1.2, 4.6, 1.1, f"context vector\n[{shown}]", GOOD, mono=True, lw=2.4)
tag(ax, 5, 0.25, "a wider arrow means a bigger share", MUTED, 9)
plt.show()

### ❓ Your turn: what is the context vector of `ν`?

We did the three steps for `δ`, the third letter. Do them for `ν`, the second one: score, softmax, blend.

👉 **Predict first:** `ν` is the row `[0, 1]`. Which of the three letters will it give the smallest share?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: the context vector of ν (row 1 of X): scores, then softmax, then the blend
# 💡 hint: for δ the lines were X @ X[2], torch.softmax(scores, dim=0), weights @ X
nu_context = None

In [ ]:
from homer_ai.lessons import check

check(
    "the context vector of ν",
    nu_context,
    torch.softmax(X @ X[1], dim=0) @ X,
    coach="Three things to look at. ν is X[1], not X[2] (we count from 0). "
    "Softmax goes between the scores and the blend. And the blend is weights @ X, "
    "which gives 2 numbers, like one row of X.",
)

<details><summary>💡 Hint</summary>

Copy the three lines of steps 1 to 3 and change one thing: the row we score against is `X[1]`.

</details>

<details><summary>✅ Answer</summary>

```python
nu_scores = X @ X[1]
nu_weights = torch.softmax(nu_scores, dim=0)
nu_context = nu_weights @ X
```

About `[0.58, 0.84]`. The scores are `[0, 1, 1]`: `ν` has nothing in common with `ἄ`, so `ἄ` gets the smallest share (0.16), and `ν` and `δ` get 0.42 each. The blend is mostly `ν` and `δ`, and both of those have a 1 in the second place, so that's the bigger number.

</details>

### ❓ Can every letter do this at once?

Yes. Every letter takes a turn as the one doing the looking, and row `i` of the table is letter `i` looking at all the letters. That's just `X @ X.T` (every row against every row, as in [A_03](../A_toolbox/A_03_dot_product.ipynb)), softmax along each row, then `weights @ X`.

One thing isn't right yet, though: `ἄ` is also looking at `ν` and `δ`, which come **after** it. When the model is guessing, those letters are in the future. [03_03](03_03_hiding_the_future.ipynb) hides them. Here we let every letter see everything, to keep it simple.

👉 **Predict first:** the last row should be the one we just did. Will it match `[0.212, 0.212, 0.576]`?

In [ ]:
all_scores = X @ X.T
all_weights = torch.softmax(all_scores, dim=-1)  # dim=-1: along each row
all_context = all_weights @ X

print("scores")
print(all_scores)
print("weights (each row adds up to 1)")
print(all_weights)
print("context vectors, one row per letter")
print(all_context)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The last row of the weights is `[0.2119, 0.2119, 0.5761]`, the same as before, and every row adds up to 1. `dim=-1` means "along the last direction", which here is along each row.

</details>

## 5. Full example: the real embeddings of `ἄνδρα`

Now the real thing: the five letters of `ἄνδρα` ("man"), each with its real 32-number embedding row from our trained model (still no learned attention, we only use `model.embedding`). The picture is a **heatmap**, a table drawn as coloured boxes. Row = the letter that's looking, column = the letter it looks at, and the number is its attention weight. The code has exactly the shape of what we did by hand.

In [ ]:
import torch

from homer_ai.lessons import load_homer_model

# read only: we load the saved model, we never save
tokenizer, model = load_homer_model(fallback_untrained=True)

In [ ]:
ids = torch.tensor(tokenizer.encode("ἄνδρα"))
with torch.no_grad():
    X = model.embedding(ids)  # shape (5, 32): the real rows, not invented ones
    scores = X @ X.T
    weights = torch.softmax(scores, dim=-1)
    context = weights @ X

print("X", tuple(X.shape), " scores", tuple(scores.shape))
print("context", tuple(context.shape))
print("each row adds up to:", weights.sum(dim=-1))
print("weight each letter gives itself:", weights.diagonal())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`X` is `(5, 32)`, the scores are `(5, 5)`, and the context is `(5, 32)`: one blended row per letter, as long as the embedding. Each row of weights adds up to 1.

</details>

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.lessons import draw_weights  # our helper that draws one heatmap
from homer_ai.studio_style import studio_figure

In [ ]:
fig = studio_figure("Simple attention", "Every letter looks mostly at itself", (8, 6.5))
letters = list("ἄνδρα")
draw_weights(
    fig, (0.14, 0.08, 0.66, 0.62), weights.numpy(), letters,
    "attention weights: each row adds up to 1",
)
plt.show()
biggest = [letters[j] for j in weights.argmax(dim=-1)]
print("each letter's biggest weight is on:", biggest)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The biggest weight in every row sits on the diagonal, so each letter mostly looks at **itself** (the printed weights on the diagonal are all above 0.9 for the trained model). Why? A row dotted with itself is its squared length, and here that tends to be bigger than its dot product with any other row.

</details>

### ❓ One quick catch: is that useful?

Not very. If every letter mostly listens to itself, the "mix" is nearly just the letter again, and nothing from the earlier letters comes through. We want to listen to the letters that **help**, not the ones that merely look the same.

The way out is to let the model learn what to look for, and that's [03_02](03_02_trainable_attention.ipynb).

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why softmax? Why not just divide each score by the total?

Scores can be negative (see the red boxes in [A_03](../A_toolbox/A_03_dot_product.ipynb)), and then dividing by the total makes no sense. Softmax makes every score positive first, so the shares always work out.

### ❓ Is the context vector the same size as the embedding?

Yes. 32 numbers per letter in, 32 numbers per letter out. That's why the output can go straight into the next layer.

## 6. Recap

- Simple attention: score every letter with a dot product, softmax the scores into weights that add up to 1, then mix the rows by those weights.
- The result is one context vector per letter, as long as the embedding (32 numbers for us).
- With nothing to learn, every letter mostly looks at itself, so the model needs to learn what to look for.

**You can now:**

- score letters against each other with a dot product
- turn scores into shares with softmax, and blend the rows into a context vector
- do it for every letter at once with `X @ X.T`

**Next:** [03_02: Trainable attention](03_02_trainable_attention.ipynb): query, key and value.